# 13 · Receive images from Reconyx cameras

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microsoft/SPARROW-Playground/blob/main/notebooks/13_comm_reconyx.ipynb)

Reconyx cellular cameras send images through the Reconyx cloud, which forwards each image to a
URL you register. sparrow-comm provides that endpoint: the forwarder logs in with a project token,
gets a short-lived JWT, then posts base64 images with the camera serial number and capture time.

This notebook plays the Reconyx forwarder and runs MegaDetector on what arrived.

In [ ]:
# Setup: install sparrow-engine if it is missing (for example on Colab).
# For an NVIDIA GPU, install `sparrow-engine-gpu` instead (same Python API).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("sparrow_engine") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sparrow-engine[video]>=0.1.31"])

import sparrow_engine as spe

print("sparrow-engine", spe.__version__, "| device:", spe.active_device())

In [ ]:
# Demo data: downloaded once into ./data and checked against a published checksum.
import hashlib
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

DATA = Path(os.environ.get("SPP_DATA_DIR", "data")).resolve()
ASSETS = {
    # Pytorch-Wildlife demo assets, CC-BY-4.0, https://zenodo.org/records/15376499
    "demo_data_det.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_det.zip?download=1",
        "md5", "373e15c8e7c8894379a82086a9177763",
    ),
    "demo_data_base.zip": (
        "https://zenodo.org/records/15376499/files/demo_data_base.zip?download=1",
        "md5", "f89b9b76c753d90e4296d62d71cf1732",
    ),
    # "Yellow-Throated Vireo Song" by SwampVids, CC-BY-4.0, Wikimedia Commons
    "vireo.wav": (
        "https://upload.wikimedia.org/wikipedia/commons/4/4f/"
        "Vireo_flavifrons_-_Yellow-Throated_Vireo_Song.wav",
        "sha1", "2b472209c1b18bfc0c225975838e3a34c1e9887a",
    ),
}


def fetch(name):
    """Download an asset (once), verify it, and unzip archives into DATA."""
    url, algo, digest = ASSETS[name]
    target = DATA / name
    DATA.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        print(f"downloading {name} ...")
        part = target.with_name(target.name + ".part")
        req = urllib.request.Request(url, headers={"User-Agent": "sparrow-playground"})
        with urllib.request.urlopen(req) as r, open(part, "wb") as f:
            shutil.copyfileobj(r, f)
        part.rename(target)
    h = hashlib.new(algo)
    with open(target, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    if h.hexdigest() != digest:
        target.unlink()
        raise RuntimeError(f"{name}: checksum mismatch, deleted; run the cell again")
    marker = DATA / f".{name}.extracted"
    if name.endswith(".zip") and not marker.exists():
        with zipfile.ZipFile(target) as z:
            z.extractall(DATA)
        marker.touch()
    return target

In [ ]:
# Setup: install sparrow-comm (the field-data receivers) if it is missing. It needs Python 3.12+.
# To try a build that is not on PyPI, set SPARROW_COMM_SPEC before starting Jupyter, e.g.
#   SPARROW_COMM_SPEC="/path/to/sparrow_comm-0.1.0-py3-none-any.whl[ftps]"
import os

if sys.version_info < (3, 12):
    raise RuntimeError("sparrow-comm needs Python 3.12 or newer, this kernel runs " + sys.version.split()[0])
for module, spec in [("sparrow_comm", os.environ.get("SPARROW_COMM_SPEC", "sparrow-comm[ftps]>=0.1.0")),
                     ("pyftpdlib", "pyftpdlib>=2.0"), ("OpenSSL", "pyopenssl>=24"), ("httpx", "httpx")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])

import sparrow_comm
from sparrow_comm.store import Store

print("sparrow-comm", sparrow_comm.__version__)

In [ ]:
# A fresh workspace for this notebook: a SQLite database plus a folder of received files.
# Store implements all three sparrow-comm hooks (registry, config, sink). In an app such as
# Sparrow Studio, the app's own database implements them instead.
WORK = Path("comm_workspace") / "13_reconyx"
shutil.rmtree(WORK, ignore_errors=True)
store = Store(WORK / "comm.db", WORK / "received")
PROJECT = 1

fetch("demo_data_det.zip")
IMAGES = sorted((DATA / "data/data_example/images").rglob("*.jpg"))[:4]
print(len(IMAGES), "demo images to send:", [p.name for p in IMAGES])

## Start the Reconyx endpoint


`TestClient` calls the FastAPI app in-process, so nothing listens on a port. A real device
sends the same HTTP requests to a running server; see *Run it for real* at the end.

In [ ]:
import secrets

from fastapi import FastAPI
from fastapi.testclient import TestClient
from sparrow_comm.reconyx import make_reconyx_router

app = FastAPI()
app.include_router(make_reconyx_router(config=store, sink=store, jwt_secret=secrets.token_hex(32)))
api = TestClient(app)

## Log in

The operator issues a token for the project and enters `proj_<id>` and the token in the Reconyx portal.

In [ ]:
token = store.issue_reconyx_key(PROJECT)
r = api.post("/v1/reconyx/login", json={"Id": f"proj_{PROJECT}", "Token": token})
jwt = r.json()["token"]
print(r.status_code, jwt[:24] + "...")

r = api.post("/v1/reconyx/login", json={"Id": f"proj_{PROJECT}", "Token": "wrong"})
print("wrong token:", r.status_code, r.json())

## Forward images

In [ ]:
import base64

for i, img in enumerate(IMAGES):
    body = {
        "Camera_serial_number": "HF2X-0042",
        "Image": base64.b64encode(img.read_bytes()).decode(),
        "Metadata": {"File_name": img.name, "date_time_taken": f"2026-01-0{i + 1}T06:30:00+02:00"},
    }
    r = api.post("/v1/reconyx/images", json=body, headers={"api_key": jwt})
    print(r.status_code, r.json())

The capture time is converted to UTC; the camera serial number becomes the camera name.

In [ ]:
import pandas as pd

received = [r for r in store.items(PROJECT) if r["kind"] == "image"]
pd.DataFrame(received)[["id", "channel", "camera_name", "original_name", "captured_at", "path"]]

## Process with sparrow-engine

In [ ]:
# Models are installed into SPARROW_ENGINE_MODEL_DIR (default ~/.sparrow-engine/models).
# Already-installed models are skipped. Inference never downloads anything by itself.
for model_id in ["MDV6-yolov10-c"]:
    spe.download_model(model_id)

In [ ]:
# Run MegaDetector V6 on the files exactly where sparrow-comm stored them.
paths = [store.data_dir / r["path"] for r in received]
results = spe.detect(paths, model="MDV6-yolov10-c", threshold=0.3)
by_path = {Path(r.path).resolve(): r for r in results}

rows = []
for r, p in zip(received, paths):
    dets = by_path[p.resolve()].detections
    rows.append({
        "channel": r["channel"],
        "camera": r["camera_name"],
        "file": r["original_name"],
        "captured_at": r["captured_at"],
        "detections": len(dets),
        "labels": sorted({d.label for d in dets}),
        "max_conf": round(max((d.confidence for d in dets), default=0.0), 2),
    })
pd.DataFrame(rows)

In [ ]:
from IPython.display import Image, display

for png in spe.visualize(results[:2]):
    display(Image(data=png, width=480))

## Run it for real

```bash
sparrow-comm --db comm.db --data-dir received reconyx issue 1
SPARROW_COMM_JWT_SECRET=$(openssl rand -hex 32) \
  sparrow-comm --db comm.db --data-dir received serve api --port 8000
```

Register `https://<your-host>/v1/reconyx` in the Reconyx portal. It must be reachable over HTTPS.